# dNATY Worker — GPU T4

**Deixa esse notebook rodando.** Ele fica escutando jobs dos seus clientes e treina na GPU T4.

- Vai em **Runtime → Change runtime type → T4 GPU** antes de rodar.
- Colab free: ~3h de T4/dia. Colab Pro: ~24h.
- Quando acabar o tempo, abra de novo e rode o notebook inteiro novamente.

In [ ]:
# ── CONFIGURAÇÃO — preencha aqui ──────────────────────────────────────────────
RAILWAY_URL  = 'https://dnaty-saas-production.up.railway.app'
WORKER_KEY   = 'COLOCA_AQUI_O_WORKER_API_KEY'  # mesmo valor que WORKER_API_KEY no Railway
REPO_URL     = 'https://github.com/pedrovergueiro/dnaty-saas.git'
POLL_SECONDS = 12  # intervalo entre polls quando não tem job

In [ ]:
# ── SETUP ─────────────────────────────────────────────────────────────────────
import subprocess, sys, os, shutil, torch

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — troca para T4 em Runtime → Change runtime type')
assert torch.cuda.is_available(), 'Precisa de GPU T4. Va em Runtime → Change runtime type → T4 GPU'

# Clona ou atualiza o repo
if not os.path.exists('/content/dnaty-saas'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/dnaty-saas'], check=True)
else:
    subprocess.run(['git', '-C', '/content/dnaty-saas', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/content/dnaty-saas', 'reset', '--hard', 'origin/main'], check=True)

# Apaga cache .pyc para forçar recompilação do código novo
for root, dirs, files in os.walk('/content/dnaty-saas'):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

# Garante caminho certo e limpa modulos em cache
if '/content/dnaty-saas' not in sys.path:
    sys.path.insert(0, '/content/dnaty-saas')

for mod in list(sys.modules.keys()):
    if mod.startswith('dnaty'):
        del sys.modules[mod]

# Importa e confirma versao
import inspect
from dnaty.evolution.evolver import DnatyEvolver
import dnaty.evolution.evolver as _ev
tem_callback = 'progress_callback' in inspect.signature(DnatyEvolver.run).parameters
print(f'Setup OK | Torch: {torch.__version__} | progress_callback: {tem_callback}')
print(f'Carregando de: {_ev.__file__}')

In [ ]:
# ── WORKER LOOP — deixa rodando ───────────────────────────────────────────────
import sys, time, requests, inspect, torch
from torchvision import datasets, transforms

for mod in list(sys.modules.keys()):
    if mod.startswith('dnaty'):
        del sys.modules[mod]

from dnaty.evolution.evolver import DnatyEvolver

DEVICE  = 'cuda'
HEADERS = {'x-worker-key': WORKER_KEY}

# Monkey-patch: adiciona progress_callback ao run() se nao tiver
if 'progress_callback' not in inspect.signature(DnatyEvolver.run).parameters:
    _original_run = DnatyEvolver.run
    def _run_with_callback(self, train_loader, val_loader,
                           early_stop_patience=8, early_stop_min_delta=1e-4,
                           progress_callback=None):
        if progress_callback is None:
            return _original_run(self, train_loader, val_loader,
                                 early_stop_patience, early_stop_min_delta)
        # injeta callback via subclasse temporaria do GenerationLog
        _orig_append = self.history.append
        def _append_with_cb(log):
            _orig_append(log)
            try: progress_callback(log)
            except Exception: pass
        self.history.append = _append_with_cb
        result = _original_run(self, train_loader, val_loader,
                               early_stop_patience, early_stop_min_delta)
        self.history.append = _orig_append  # restaura
        return result
    DnatyEvolver.run = _run_with_callback
    print('progress_callback: injetado via patch')
else:
    print('progress_callback: nativo')


def load_dataset(name):
    dataset_map = {
        'mnist':         (datasets.MNIST,        (0.5,),        (0.5,),        784,  10),
        'fashion_mnist': (datasets.FashionMNIST,  (0.5,),        (0.5,),        784,  10),
        'cifar10':       (datasets.CIFAR10,       (0.5,0.5,0.5), (0.5,0.5,0.5), 3072, 10),
    }
    if name not in dataset_map:
        raise ValueError(f'Dataset {name} nao suportado')
    cls, mean, std, input_size, n_classes = dataset_map[name]
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    kw = dict(batch_size=512, num_workers=2, pin_memory=True)
    train = torch.utils.data.DataLoader(cls('/tmp/data', train=True,  download=True, transform=t), shuffle=True,  **kw)
    test  = torch.utils.data.DataLoader(cls('/tmp/data', train=False, download=True, transform=t), shuffle=False, **kw)
    return train, test, input_size, n_classes


def send_progress(job_id, log):
    try:
        requests.post(
            f'{RAILWAY_URL}/api/v1/worker/jobs/{job_id}/progress',
            headers=HEADERS,
            json={
                'generation': log.gen,
                'best_acc':   float(log.best_acc),
                'delta_grad': float(log.delta_grad),
                'delta_mem':  float(log.delta_mem),
                'n_params':   int(log.n_params),
            },
            timeout=6,
        )
    except Exception as e:
        print(f'  [progress warn] {e}')


def process_job(job_id, params):
    print(f'\n>>> JOB {job_id}')
    print(f'    dataset={params["dataset"]} | pop={params["n_pop"]} | gens={params["n_generations"]} | device={DEVICE}')

    train_loader, test_loader, input_size, n_classes = load_dataset(params['dataset'])

    evolver = DnatyEvolver(
        n_pop=params['n_pop'],
        n_generations=params['n_generations'],
        t_local=params['t_local'],
        lr=params['lr'],
        lambda1=params['lambda1'],
        lambda2=params['lambda2'],
        device=DEVICE,
        input_size=input_size,
        n_classes=n_classes,
        init_hidden=params['init_hidden'],
        batch_size=params['batch_size'],
        verbose=True,
    )

    t0 = time.time()
    best, _ = evolver.run(
        train_loader, test_loader,
        progress_callback=lambda log: send_progress(job_id, log),
    )
    duration = time.time() - t0

    model = best.model
    requests.post(
        f'{RAILWAY_URL}/api/v1/worker/jobs/{job_id}/complete',
        headers=HEADERS,
        json={
            'best_accuracy':    float(best.acc),
            'layer_sizes':      list(model.layer_sizes),
            'activations':      list(model.activations),
            'n_params':         int(sum(p.numel() for p in model.parameters())),
            'duration_seconds': duration,
        },
        timeout=10,
    )
    print(f'<<< CONCLUIDO {job_id} | acc={best.acc:.4f} | {duration:.0f}s')


# ── Loop principal ─────────────────────────────────────────────────────────────
print(f'Worker iniciado. Polling a cada {POLL_SECONDS}s...')
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory//1024**3}GB')

while True:
    try:
        r = requests.get(f'{RAILWAY_URL}/api/v1/worker/next-job', headers=HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()

        if not data.get('job_id'):
            print('.', end='', flush=True)
            time.sleep(POLL_SECONDS)
            continue

        job_id = data['job_id']
        params = data['params']

        try:
            process_job(job_id, params)
        except Exception as exc:
            print(f'\n[FALHOU] {job_id}: {exc}')
            try:
                requests.post(
                    f'{RAILWAY_URL}/api/v1/worker/jobs/{job_id}/fail',
                    headers=HEADERS,
                    json={'error': str(exc)},
                    timeout=6,
                )
            except Exception:
                pass

    except KeyboardInterrupt:
        print('\nWorker parado.')
        break
    except Exception as e:
        print(f'\n[ERRO poll] {e}')
        time.sleep(POLL_SECONDS)